# 04 – A stronger model

**Goal:** beat the best baseline from notebook 03 (logistic regression on 38 features: **PR-AUC 0.383** on validation) with a gradient-boosted tree model (LightGBM), then run the checks planned in the README.

**Update after the literature review:** published models of trial termination found **eligibility-criteria complexity** among the strongest registry signals. Three such features were added to `analysis.features` (word count, number of inclusion criteria, number of exclusion criteria), giving **41 features**. This was done before any test-set scoring, and everything below uses the 41 features.

1. Load the data (same features and splits as notebook 03)
2. Refit the baseline to beat
3. LightGBM with default-ish settings
4. A small tuning grid, **scored on validation only**
5. Results table
6. What the model relies on (feature importance)
7. Checks: trials already recruiting in 2018, the unknown-status sensitivity label, calibration, and the `overdue_2018` surprise
8. Save the chosen settings for notebook 05

**The test set (2018 registrations) is still not used.** Picking the best of several settings on validation makes the validation score slightly optimistic, which is exactly why test exists: notebook 05 scores the chosen model on it once.

## 1. Load the data

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import json
from itertools import product

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from src.evaluate import evaluate
from src.features import (ALL_FEATURES, BINARY, CATEGORICAL, LOG_NUMERIC, NUMERIC,
                          load_features, split_xy)
from src.preprocess import make_preprocessor, make_tree_preprocessor

df = load_features()  # labelled trials only
X_train, y_train = split_xy(df, "train")
X_valid, y_valid = split_xy(df, "valid")

print(f"labelled trials: {len(df):,}")
print(f"train: {len(X_train):,} trials, {y_train.mean():.1%} stopped early")
print(f"valid: {len(X_valid):,} trials, {y_valid.mean():.1%} stopped early")
assert (len(X_train), len(X_valid)) == (11776, 9157)

labelled trials: 31,903
train: 11,776 trials, 20.9% stopped early
valid: 9,157 trials, 20.2% stopped early


## 2. The baseline to beat

The same logistic regression as notebook 03, now with all 41 features, refitted here so that every model in this notebook's results table is scored in the same run. (With 38 features it scored PR-AUC 0.383.)

In [2]:
results = []

logreg = Pipeline([
    ("prep", make_preprocessor(categorical=CATEGORICAL, log_numeric=LOG_NUMERIC,
                               numeric=NUMERIC, binary=BINARY)),
    ("model", LogisticRegression(max_iter=2000)),
])
logreg.fit(X_train, y_train)
valid_prob_logreg = logreg.predict_proba(X_valid)[:, 1]

results.append(evaluate(y_valid, valid_prob_logreg, "logistic regression (all features)"))
results[-1]

{'model': 'logistic regression (all features)',
 'n_trials': 9157,
 'positive_rate': 0.20192202686469368,
 'pr_auc': 0.38324485284259,
 'roc_auc': 0.6959124934172024,
 'brier': 0.14826245544356353}

## 3. LightGBM with standard settings

**Gradient boosting** builds many small decision trees one after another, each correcting the errors of the previous ones. Unlike logistic regression it can capture **interactions** (e.g. "industry sponsor *and* not yet recruiting") and **non-linear effects** (e.g. risk rising only above a certain enrolment target) without us creating them by hand.

Trees don't need scaling or log transforms, and LightGBM handles missing numbers itself, so it gets a lighter preprocessor: only the text columns are one-hot encoded (`make_tree_preprocessor` in `src/preprocess.py`).

In [3]:
def make_lgbm(num_leaves=31, min_child_samples=20, n_estimators=300, learning_rate=0.05):
    """A LightGBM pipeline. The fixed settings add a little randomness to reduce overfitting."""
    return Pipeline([
        ("prep", make_tree_preprocessor(CATEGORICAL)),
        ("model", LGBMClassifier(
            num_leaves=num_leaves,               # max leaves per tree (tree complexity)
            min_child_samples=min_child_samples, # min trials in a leaf (bigger = smoother)
            n_estimators=n_estimators,           # number of trees
            learning_rate=learning_rate,         # how much each tree corrects the previous ones
            subsample=0.8, subsample_freq=1,     # each tree sees 80% of the trials
            colsample_bytree=0.8,                # ... and 80% of the columns
            random_state=42,
            verbose=-1,
        )),
    ])


lgbm_default = make_lgbm()
lgbm_default.fit(X_train, y_train)
valid_prob_default = lgbm_default.predict_proba(X_valid)[:, 1]

results.append(evaluate(y_valid, valid_prob_default, "LightGBM (standard settings)"))
results[-1]

{'model': 'LightGBM (standard settings)',
 'n_trials': 9157,
 'positive_rate': 0.20192202686469368,
 'pr_auc': 0.37839150740012195,
 'roc_auc': 0.6952261655363052,
 'brier': 0.14919875738018706}

## 4. A small tuning grid (validation only)

18 combinations of the three settings that matter most. Each is trained on `train` and scored on `valid`. The grid is kept small on purpose: the more combinations we try, the more the best validation score is inflated by luck.

In [4]:
grid = {
    "num_leaves": [15, 31, 63],
    "min_child_samples": [20, 50, 100],
    "n_estimators": [200, 500],
}

tuning = []
for num_leaves, min_child_samples, n_estimators in product(*grid.values()):
    model = make_lgbm(num_leaves=num_leaves, min_child_samples=min_child_samples,
                      n_estimators=n_estimators)
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_valid)[:, 1]
    tuning.append({"num_leaves": num_leaves, "min_child_samples": min_child_samples,
                   "n_estimators": n_estimators,
                   **evaluate(y_valid, prob, "LightGBM")})

tuning_table = (pd.DataFrame(tuning)
                .drop(columns=["model", "n_trials", "positive_rate"])
                .sort_values("pr_auc", ascending=False)
                .round(3))
display(tuning_table)

,num_leaves,min_child_samples,n_estimators,pr_auc,roc_auc,brier
0,15,20,200,0.391,0.708,0.146
4,15,100,200,0.388,0.705,0.147
10,31,100,200,0.388,0.701,0.147
2,15,50,200,0.388,0.705,0.147
6,31,20,200,0.383,0.699,0.148
8,31,50,200,0.383,0.699,0.148
1,15,20,500,0.381,0.696,0.149
5,15,100,500,0.380,0.698,0.149
3,15,50,500,0.380,0.693,0.149
16,63,100,200,0.379,0.690,0.149


### Refit the best combination

In [5]:
best = tuning_table.iloc[0]
best_params = {k: int(best[k]) for k in grid}
print("best settings:", best_params)

lgbm_best = make_lgbm(**best_params)
lgbm_best.fit(X_train, y_train)
valid_prob_best = lgbm_best.predict_proba(X_valid)[:, 1]

results.append(evaluate(y_valid, valid_prob_best, "LightGBM (tuned)"))
results[-1]

best settings: {'num_leaves': 15, 'min_child_samples': 20, 'n_estimators': 200}


{'model': 'LightGBM (tuned)',
 'n_trials': 9157,
 'positive_rate': 0.20192202686469368,
 'pr_auc': 0.3905565821532715,
 'roc_auc': 0.7079039158728088,
 'brier': 0.14633141338998049}

## 5. Results on the validation set

For reference, "no model" scores PR-AUC 0.202 and ROC-AUC 0.500 (notebook 03).

In [6]:
results_table = pd.DataFrame(results).set_index("model").round(3)
display(results_table[["pr_auc", "roc_auc", "brier"]])

gain = results_table.loc["LightGBM (tuned)", "pr_auc"] - results_table.loc["logistic regression (all features)", "pr_auc"]
print(f"LightGBM (tuned) vs logistic regression: {gain:+.3f} PR-AUC")

,pr_auc,roc_auc,brier
model,,,
logistic regression (all features),0.383,0.696,0.148
LightGBM (standard settings),0.378,0.695,0.149
LightGBM (tuned),0.391,0.708,0.146


LightGBM (tuned) vs logistic regression: +0.008 PR-AUC


## 6. What does LightGBM rely on? Feature importance

**Gain** is the total improvement a feature brought to the trees' splits, shown here as a share of the total. It says how much the model *uses* a feature, not whether the feature raises or lowers the risk, and it's not a causal effect.

One-hot columns (e.g. `status_2018_RECRUITING`) appear separately.

In [7]:
booster = lgbm_best.named_steps["model"]
importance = pd.Series(
    booster.booster_.feature_importance(importance_type="gain"),
    index=lgbm_best.named_steps["prep"].get_feature_names_out(),
)
importance = (importance / importance.sum()).sort_values(ascending=False)

display(importance.head(15).round(3).rename("share_of_gain"))
print(f"Top 10 features account for {importance.head(10).sum():.0%} of the gain")

status_2018_NOT_YET_RECRUITING    0.142
months_registration_to_start      0.119
planned_duration_months           0.091
enrollment_target                 0.075
criteria_words                    0.056
is_behavioral                     0.053
max_age_years                     0.050
n_exclusion_criteria              0.040
n_sites                           0.040
n_inclusion_criteria              0.027
min_age_years                     0.025
n_collaborators                   0.023
has_us_site                       0.022
status_2018_RECRUITING            0.019
n_interventions                   0.015
Name: share_of_gain, dtype: float64

Top 10 features account for 69% of the gain


## 7. Checks

### 7a. Trials already recruiting in 2018

The same robustness check as notebook 03: does the model still work when "not yet recruiting → withdrawn" can't help? The comparison point is this subgroup's own stop rate.

In [8]:
recruiting = (X_valid["status_2018"] == "RECRUITING").to_numpy()

subgroup = pd.DataFrame([
    evaluate(y_valid[recruiting], valid_prob_logreg[recruiting], "logistic regression, RECRUITING only"),
    evaluate(y_valid[recruiting], valid_prob_best[recruiting], "LightGBM (tuned), RECRUITING only"),
]).set_index("model").round(3)
display(subgroup)

,n_trials,positive_rate,pr_auc,roc_auc,brier
model,,,,,
"logistic regression, RECRUITING only",8050,0.173,0.282,0.663,0.136
"LightGBM (tuned), RECRUITING only",8050,0.173,0.298,0.678,0.135


### 7b. Sensitivity check: counting unknown-status trials as stopped

24% of the cohort ended with *unknown* status and is excluded from the main label. Many were probably abandoned. Here the tuned model is retrained with `label_sensitivity`, where **unknown = stopped**. The base rate is much higher, so compare each PR-AUC with its own positive rate. What matters is whether the model still clearly beats it.

In [9]:
df_sens = load_features(label_column="label_sensitivity")
X_train_s, y_train_s = split_xy(df_sens, "train", label_column="label_sensitivity")
X_valid_s, y_valid_s = split_xy(df_sens, "valid", label_column="label_sensitivity")

lgbm_sens = make_lgbm(**best_params)
lgbm_sens.fit(X_train_s, y_train_s)
valid_prob_sens = lgbm_sens.predict_proba(X_valid_s)[:, 1]

sensitivity = pd.DataFrame([
    evaluate(y_valid, valid_prob_best, "main label (unknown excluded)"),
    evaluate(y_valid_s, valid_prob_sens, "sensitivity label (unknown = stopped)"),
]).set_index("model").round(3)
display(sensitivity)

,n_trials,positive_rate,pr_auc,roc_auc,brier
model,,,,,
main label (unknown excluded),9157,0.202,0.391,0.708,0.146
sensitivity label (unknown = stopped),12882,0.433,0.660,0.724,0.209


### 7c. Calibration: are the probabilities trustworthy?

The validation trials are split into 10 equal-sized groups by predicted risk. In each group, the **average predicted probability** should be close to the **share that actually stopped**. If it is, a prediction of "30%" really means about 3 in 10.

In [10]:
observed, predicted = calibration_curve(y_valid, valid_prob_best, n_bins=10, strategy="quantile")
calibration = pd.DataFrame({
    "mean_predicted": predicted.round(3),
    "observed_stop_rate": observed.round(3),
})
calibration["gap"] = (calibration["observed_stop_rate"] - calibration["mean_predicted"]).round(3)
display(calibration)

,mean_predicted,observed_stop_rate,gap
0,0.051,0.038,-0.013
1,0.084,0.086,0.002
2,0.110,0.115,0.005
3,0.131,0.148,0.017
4,0.152,0.166,0.014
5,0.176,0.181,0.005
6,0.202,0.227,0.025
7,0.234,0.252,0.018
8,0.292,0.319,0.027
9,0.553,0.486,-0.067


### 7d. The `overdue_2018` surprise

In notebook 03, "still recruiting after the planned completion date" leaned towards *completed*. A simple look at the raw stop rates in `train`, split by the 2018 status, shows whether that's real or an artefact of how the features overlap.

In [11]:
train_rows = df[df["split"] == "train"]
display(train_rows.groupby(["status_2018", "overdue_2018"])["label"]
        .agg(n_trials="count", stop_rate="mean").round(3))

n_trials  stop_rate
status_2018        overdue_2018                     
NOT_YET_RECRUITING 0.0                477      0.665
                   1.0                113      0.416
RECRUITING         0.0               8786      0.197
                   1.0               2398      0.152

## 8. Save the settings for notebook 05

Decision recorded **before** the test set is used: logistic regression is the **primary** model (simpler, interpretable, and not tuned on validation), and LightGBM with the best settings above is reported alongside it. Both, with their validation scores, are saved to `reports/04_final_model.json` so that notebook 05 rebuilds exactly these models.

In [12]:
final_choice = {
    "primary_model": "logistic_regression",
    "logistic_regression_params": {"max_iter": 2000},
    "lightgbm_params": best_params,
    "n_features": len(ALL_FEATURES),
    "valid_scores": {
        "logistic_regression": results_table.loc["logistic regression (all features)", ["pr_auc", "roc_auc", "brier"]].to_dict(),
        "lightgbm": results_table.loc["LightGBM (tuned)", ["pr_auc", "roc_auc", "brier"]].to_dict(),
    },
}

output_path = Path.cwd().parent / "reports" / "04_final_model.json"
output_path.write_text(json.dumps(final_choice, indent=2))
print(json.dumps(final_choice, indent=2))

{
  "primary_model": "logistic_regression",
  "logistic_regression_params": {
    "max_iter": 2000
  },
  "lightgbm_params": {
    "num_leaves": 15,
    "min_child_samples": 20,
    "n_estimators": 200
  },
  "n_features": 41,
  "valid_scores": {
    "logistic_regression": {
      "pr_auc": 0.383,
      "roc_auc": 0.696,
      "brier": 0.148
    },
    "lightgbm": {
      "pr_auc": 0.391,
      "roc_auc": 0.708,
      "brier": 0.146
    }
  }
}


## 9. Findings

| Model (validation, 9,157 trials, 20.2% stopped) | PR-AUC | ROC-AUC | Brier |
|---|---|---|---|
| logistic regression, all 41 features | 0.383 | 0.696 | 0.148 |
| LightGBM, standard settings | 0.378 | 0.695 | 0.149 |
| LightGBM, tuned (15 leaves, min 20 per leaf, 200 trees) | 0.391 | 0.708 | 0.146 |

- **LightGBM barely beats logistic regression** (+0.008 PR-AUC), and picking the best of 18 settings on validation inflates even that. The best settings are the simplest trees. The signal in registry features is **mostly simple and additive**.
- **The 3 eligibility-criteria features** (added after the literature review) are used, with word count the 5th most important feature, but they **didn't improve the scores**: logistic regression 0.383 → 0.383, LightGBM 0.389 → 0.391 compared with the 38-feature run.
- **What LightGBM relies on:** not yet recruiting in 2018, the gap between registration and start, planned duration, target enrolment, criteria word count, behavioural interventions, and maximum age.
- **Trials already recruiting in 2018:** PR-AUC 0.298 (LightGBM) / 0.282 (logistic regression) vs a base rate of 0.173.
- **Sensitivity (unknown = stopped):** PR-AUC 0.660 vs a base rate of 0.433, so the conclusions don't depend on how unknown trials are handled.
- **Calibration on validation is good** (within ~3 points in 9 of 10 groups). The riskiest 10% is overestimated (55% predicted vs 49% observed).
- **`overdue_2018` is real:** overdue trials stopped less often within both 2018 statuses (not yet recruiting: 41.6% vs 66.5%; recruiting: 15.2% vs 19.7%).

**Decision recorded before the test:** logistic regression is the primary model; LightGBM is also reported (settings saved in `reports/04_final_model.json`).